In [1]:
from nba_api.stats.endpoints import playercareerstats
from nba_api.stats.static import players
import pandas as pd

# Find LeBron James in the NBA database
lebron = players.find_players_by_full_name("LeBron James")[0]
print("Player found:", lebron)

Player found: {'id': 2544, 'full_name': 'LeBron James', 'first_name': 'LeBron', 'last_name': 'James', 'is_active': True}


In [2]:
# Use LeBron's ID to fetch his entire career stats
career = playercareerstats.PlayerCareerStats(player_id=2544)

# The NBA API returns data in a special format
# .get_data_frames() converts it into a pandas DataFrame
# Think of it as converting raw data into a clean spreadsheet
df = career.get_data_frames()[0]

# Show the shape of the data
# This tells us how many rows and columns we have
print("Shape:", df.shape)

# Show the column names
print("\nColumns:", list(df.columns))

Shape: (23, 27)

Columns: ['PLAYER_ID', 'SEASON_ID', 'LEAGUE_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'PLAYER_AGE', 'GP', 'GS', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']


In [3]:
# Show the last 5 seasons of LeBron's career
# tail(5) means "give me the last 5 rows"
# Think of it like scrolling to the bottom of a spreadsheet

df_clean = df[['SEASON_ID', 'TEAM_ABBREVIATION', 'GP', 'PTS', 'AST', 'REB', 'FG_PCT']].tail(5)
print(df_clean.to_string())

   SEASON_ID TEAM_ABBREVIATION  GP   PTS  AST  REB  FG_PCT
18   2021-22               LAL  56  1695  349  459   0.524
19   2022-23               LAL  55  1590  375  457   0.500
20   2023-24               LAL  71  1822  589  518   0.540
21   2024-25               LAL  70  1710  575  546   0.513
22   2025-26               LAL  57  1184  403  346   0.512


In [4]:
# Calculate points per game for each season
# This is your first data transformation in pandas

# Create a new column called PPG (points per game)
# We divide total points by games played for each row
# pandas does this for every row automatically, no loop needed
df['PPG'] = df['PTS'] / df['GP']
df['APG'] = df['AST'] / df['GP']
df['RPG'] = df['REB'] / df['GP']

# Round to 1 decimal place to make it readable
df['PPG'] = df['PPG'].round(1)
df['APG'] = df['APG'].round(1)
df['RPG'] = df['RPG'].round(1)

# Show the last 5 seasons with our new calculated columns
result = df[['SEASON_ID', 'GP', 'PPG', 'APG', 'RPG', 'FG_PCT']].tail(5)
print(result.to_string())

   SEASON_ID  GP   PPG  APG  RPG  FG_PCT
18   2021-22  56  30.3  6.2  8.2   0.524
19   2022-23  55  28.9  6.8  8.3   0.500
20   2023-24  71  25.7  8.3  7.3   0.540
21   2024-25  70  24.4  8.2  7.8   0.513
22   2025-26  57  20.8  7.1  6.1   0.512


In [5]:
# LeBron is 40 years old and still putting up 20+ PPG
# Let's find his career best and worst seasons
# and see his full scoring journey

print("=== LeBron James Career Scoring Journey ===\n")

# Find his best scoring season
best_season = df.loc[df['PPG'].idxmax()]
print(f"Best scoring season: {best_season['SEASON_ID']} with {best_season['PPG']} PPG")

# Find his current form vs career average
career_avg = df['PPG'].mean().round(1)
current_ppg = df.iloc[-1]['PPG']
diff = round(current_ppg - career_avg, 1)

print(f"Career average PPG: {career_avg}")
print(f"Current season PPG: {current_ppg}")
print(f"Difference from career avg: {diff} PPG")

# Find seasons where he scored 25+ PPG
elite_seasons = df[df['PPG'] >= 25.0]
print(f"\nSeasons with 25+ PPG: {len(elite_seasons)} out of {len(df)} total seasons")

=== LeBron James Career Scoring Journey ===

Best scoring season: 2005-06 with 31.4 PPG
Career average PPG: 26.7
Current season PPG: 20.8
Difference from career avg: -5.9 PPG

Seasons with 25+ PPG: 20 out of 23 total seasons


In [6]:
from nba_api.stats.endpoints import playergamelog
import time

# Fetch LeBron's game by game log for this season
# 2024-25 is the season ID format the NBA API uses
print("Fetching LeBron's game logs... this takes a few seconds")

# We add time.sleep to avoid hitting the API too fast
# The NBA API has rate limiting, meaning if you call it
# too many times too quickly it will block you temporarily
time.sleep(1)

gamelog = playergamelog.PlayerGameLog(
    player_id=2544,
    season='2024-25'
)

games = gamelog.get_data_frames()[0]

print(f"Games fetched: {len(games)}")
print(f"\nColumns available: {list(games.columns)}")

Fetching LeBron's game logs... this takes a few seconds
Games fetched: 70

Columns available: ['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE']


In [7]:
# Let's look at his last 10 games
# The data comes in reverse chronological order
# so the most recent games are at the top

recent = games[['GAME_DATE', 'MATCHUP', 'WL', 'PTS', 'AST', 'REB', 'FG_PCT', 'PLUS_MINUS']].head(10)
print("=== LeBron's Last 10 Games ===\n")
print(recent.to_string())

# Calculate his averages over the last 10 games
print("\n=== Last 10 Game Averages ===")
print(f"PPG:  {games['PTS'].head(10).mean().round(1)}")
print(f"APG:  {games['AST'].head(10).mean().round(1)}")
print(f"RPG:  {games['REB'].head(10).mean().round(1)}")
print(f"FG%:  {(games['FG_PCT'].head(10).mean() * 100).round(1)}%")

# Find his best and worst games this season
best_game = games.loc[games['PTS'].idxmax()]
worst_game = games.loc[games['PTS'].idxmin()]

print(f"\nBest game this season:  {best_game['PTS']} pts vs {best_game['MATCHUP']} on {best_game['GAME_DATE']}")
print(f"Worst game this season: {worst_game['PTS']} pts vs {worst_game['MATCHUP']} on {worst_game['GAME_DATE']}")

=== LeBron's Last 10 Games ===

      GAME_DATE      MATCHUP WL  PTS  AST  REB  FG_PCT  PLUS_MINUS
0  Apr 11, 2025  LAL vs. HOU  W   14    8    4   0.545           5
1  Apr 09, 2025    LAL @ DAL  W   27    3    7   0.550          18
2  Apr 08, 2025    LAL @ OKC  L   28    3    7   0.421         -23
3  Apr 06, 2025    LAL @ OKC  W   19    7    3   0.563           7
4  Apr 04, 2025  LAL vs. NOP  W   27    8    0   0.529           7
5  Apr 03, 2025  LAL vs. GSW  L   33    9    5   0.667          -1
6  Mar 31, 2025  LAL vs. HOU  W   16    4    8   0.417           9
7  Mar 29, 2025    LAL @ MEM  W   25    8    6   0.476          -1
8  Mar 27, 2025    LAL @ CHI  L   17   12    5   0.400         -10
9  Mar 26, 2025    LAL @ IND  W   13    7   13   0.333           5

=== Last 10 Game Averages ===
PPG:  21.9
APG:  6.9
RPG:  5.8
FG%:  49.0%

Best game this season:  42 pts vs LAL vs. GSW on Feb 06, 2025
Worst game this season: 10 pts vs LAL @ MIN on Dec 02, 2024


In [8]:
# Home vs Away analysis
# Look at the MATCHUP column
# "LAL vs. XXX" means HOME game
# "LAL @ XXX" means AWAY game

# Create a new column that identifies home or away
# If the matchup contains "vs." it is a home game
games['HOME_AWAY'] = games['MATCHUP'].apply(
    lambda x: 'HOME' if 'vs.' in x else 'AWAY'
)

# Group by home/away and calculate averages
# groupby is like a pivot table in Excel
# it groups rows by a category and calculates stats for each group
home_away_stats = games.groupby('HOME_AWAY')[['PTS', 'AST', 'REB', 'FG_PCT']].mean().round(2)

print("=== LeBron Home vs Away This Season ===\n")
print(home_away_stats.to_string())

# Count wins at home vs away
win_rate = games.groupby('HOME_AWAY')['WL'].apply(
    lambda x: f"{(x == 'W').sum()}W - {(x == 'L').sum()}L"
)

print("\n=== Win/Loss Record ===")
print(win_rate.to_string())

=== LeBron Home vs Away This Season ===

             PTS   AST   REB  FG_PCT
HOME_AWAY                           
AWAY       23.97  8.25  7.56    0.49
HOME       24.91  8.18  8.06    0.53

=== Win/Loss Record ===
HOME_AWAY
AWAY    19W - 17L
HOME     25W - 9L


In [9]:
# Rest days analysis
# Convert GAME_DATE to a proper date format so we can do math on it
games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])

# Sort by date oldest to newest so we can calculate gaps
games_sorted = games.sort_values('GAME_DATE').reset_index(drop=True)

# Calculate days between each game
# diff() finds the difference between consecutive rows
# dt.days converts the time difference to just the number of days
games_sorted['DAYS_REST'] = games_sorted['GAME_DATE'].diff().dt.days

# Fill the first game with 3 days rest as default
games_sorted['DAYS_REST'] = games_sorted['DAYS_REST'].fillna(3)

# Group by rest days and see how scoring changes
rest_analysis = games_sorted.groupby('DAYS_REST')[['PTS', 'FG_PCT']].agg({
    'PTS': ['mean', 'count'],
    'FG_PCT': 'mean'
}).round(2)

print("=== LeBron Performance by Rest Days ===\n")
print(rest_analysis.to_string())

=== LeBron Performance by Rest Days ===

             PTS       FG_PCT
            mean count   mean
DAYS_REST                    
1.0        25.36    11   0.51
2.0        24.74    43   0.51
3.0        24.67     9   0.54
4.0        21.50     2   0.48
6.0        20.50     2   0.58
7.0        26.00     1   0.46
9.0        18.00     1   0.41
14.0       17.00     1   0.44
